# NSD: conventional GLMs, optimized HRFs, and beta series

This notebook analyzes every run of **sub-07 / ses-nsd10** using fMRIPrep CIFTI time series.
It fits the same three-regressor GLM twice: first with the canonical SPM HRF, then with an
HRF selected separately at each grayordinate. It also produces single-trial beta series,
odd/even HRF reliability maps, encoding-guided ridge selection, and reaction-time checks.

The GLM predictors are **task**, **response_time**, and **trial_type**. Both modulators are
centered within run and fitted jointly, without orthogonalization. The observed trial-type
codes are 0 and 1; this notebook does not assign them behavioral labels.

Run from a checkout with `uv sync --group dev` and select its `.venv` Python kernel in your
notebook editor. The local data are not distributed with boldtailor. See [the NSD guide](README.md)
for input details. The helper modules beside this notebook handle CIFTI blocks and output files;
the fitting calls use boldtailor's public APIs.

## 1. Paths and execution settings

The defaults process all 12 runs and all 91,282 grayordinates, using four worker processes.
For a quick check, set `max_grayordinates=128`; the notebook still uses **every run** and
exports the original spatial axis, with NaN outside the requested subset. `block_size` controls
the number of grayordinates loaded together. Lower `n_jobs` or `block_size` to reduce memory use.

Outputs go under `derivatives/boldtailor`, with `desc-notebook...` filenames. Existing notebook
outputs cause an error before fitting; choose another output directory when comparing settings.
`ridge_mode="fractional_cv"` selects a shrinkage fraction independently at each grayordinate
using trial type and RT. `ridge_fractions` sets the candidate grid (0.1 through 1 by default).
Use `"cv"` for the earlier global-alpha selection, `"fixed"` with
`ridge_alpha` to reproduce a chosen penalty, or `"off"` for OLS alone. Older `NSD_CONFIG`
overrides containing only `ridge_alpha` still select fixed mode (`None` disables it).
Optimized-HRF ridge CV needs at least three odd and three even runs. HRF selection uses mean
stimulus prediction; neither stage requires repeated images.

Set `NSD_BIDS_ROOT` in the environment before starting the kernel, or supply
`bids_root` in a `NSD_CONFIG` dictionary before the setup cell.
Optional `NSD_FMRIPREP_ROOT` and `NSD_OUTPUT_ROOT` override derivative locations.
Explicit dictionary paths take precedence. The helpers are checkout-local examples;
Boldtailor itself models arrays without requiring NSD or a particular imaging format.


In [ ]:
from pathlib import Path
import sys

repo = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/boldtailor").is_dir() and (p / "pyproject.toml").is_file())
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from examples.NSD.notebook_paths import notebook_paths

overrides = globals().get("NSD_CONFIG", {})
settings = dict(
    subject="sub-07", session="ses-nsd10",
    n_jobs=4, block_size=4096, max_grayordinates=None,
    encoding_mode="within_run",  # "absolute" reproduces the shared-intercept objective.
    ridge_mode="fractional_cv",  # "fractional_cv", "cv" (global alpha), "fixed", or "off"
    ridge_fractions=[.1, .2, .3, .4, .5, .6, .7, .8, .9, 1.],
    ridge_alphas=[0., .001, .01, .1, 1., 10., 100.], ridge_percentile=90.,
    ridge_alpha=0.1,  # Used only in fixed mode.
    hrf_library="sobol", hrf_n_samples=512, hrf_seed=0,
    hrf_parameters=None,  # Optional custom rows override the library choice.
    surface_maps=True, surface_meshes=None,  # Optional {"left": path, "right": path} in fsLR order.
)
settings.update(overrides)
settings.update(notebook_paths(overrides))
if "ridge_mode" not in overrides and "ridge_alpha" in overrides:
    settings["ridge_mode"] = "off" if overrides["ridge_alpha"] is None else "fixed"
elif "ridge_mode" not in overrides and "ridge_alphas" in overrides:
    settings["ridge_mode"] = "cv"
if settings["ridge_mode"] not in ("fractional_cv", "cv", "fixed", "off"):
    raise ValueError("ridge_mode must be fractional_cv, cv, fixed, or off")
root, prep, output = (Path(settings[k]) for k in ("bids_root", "fmriprep_root", "output_root"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from boldtailor.hrf_library import sobol_hrf_library, expanded_hrf_library, HrfLibrary
from examples.NSD.workflow_inputs import (
    load_session, make_blocks, run_summary, glm_events, glm_model, REGRESSORS,
)
from examples.NSD.workflow_analysis import select_hrfs, selection_maps, fit_glms, fit_beta_series
from examples.NSD.workflow_outputs import check_output, save_workflow
from examples.NSD.rt_diagnostics import even_run_points
from examples.NSD.workflow_plots import (
    design_figure, library_figure, glm_comparison, parameter_agreement, curve_agreement,
)

check_output(output, settings["subject"], settings["session"])
workers = settings["n_jobs"]
figures = {}

## 2. Inspect the inputs and nuisance model

Each model includes 24 motion parameters, the six retained combined-mask aCompCor components
with the greatest explained variance, fMRIPrep's cosine high-pass columns, and a run intercept.
There is no additional drift basis, smoothing, or signal rescaling. Coefficients retain native
signal units. Conventional GLMs use OLS; beta-series fits include OLS and the selected ridge method.

We remove the leading volumes marked as nonsteady, consistently in every model. **Acquisition
times and event onsets retain their original values**, including the BOLD JSON `StartTime`.
This gives a common nuisance-column list across runs with different numbers of flagged volumes.
R² is evaluated on the retained scans; it need not equal the earlier scripts' R², which includes
nonsteady indicator regressors and all scans. Interior nonsteady flags are an error.

In [ ]:
runs = load_session(root, prep, subject=settings["subject"], session=settings["session"])
brain = runs[0].image.header.get_axis(1)
blocks = make_blocks(runs, block_size=settings["block_size"],
                     max_grayordinates=settings["max_grayordinates"])
print(f"{len(runs)} runs; {sum(len(r.events) for r in runs)} trials; "
      f"{sum(len(b) for b in blocks):,} nonconstant grayordinates to fit / {len(brain):,} total")
display(run_summary(runs))
display(runs[0].events.head())
display(pd.Series(runs[0].confounds.columns, name="Nuisance column"))

## 3. Define the conventional GLM

| Predictor | Event amplitude | Coefficient interpretation |
| --- | --- | --- |
| `task` | 1 for every presentation | Mean stimulus response at the run's mean RT and trial-type proportion |
| `response_time` | RT in seconds minus its run mean | Activation change per additional second of RT, controlling for trial type |
| `trial_type` | Binary code minus its run mean | Type 1 minus type 0, controlling for RT |

The three columns are convolved with the same HRF within a grayordinate. Centering changes
the interpretation of `task`; it does not orthogonalize the modulators. Input validation requires
finite RTs and both trial-type codes in each run. Collinear or nonestimable designs fail explicitly.

The GLM event table has three rows per presentation, one for each predictor. HRF selection and
single-trial fits instead use the **original table with one row per presentation**.

In [ ]:
model = glm_model(runs)
display(glm_events(runs[0].events).groupby("trial_type", sort=False).head(3))
print("Contrasts:", dict(model.contrasts))
print("HRF:", model.hrf_model, "| noise:", model.noise_model, "| extra drift:", model.drift_model)

## 4. Fit the canonical-HRF GLM

The block helper calls `fit(glm_data, model)` and `task_delta_r2(glm_data, model, result)`.
Contrasts combine runs with equal-run fixed effects. R² pools residual sums of squares across
runs and divides by the sum of within-run total sums of squares; it is not an average of run R².
Constant grayordinates remain at their original spatial positions as NaN.

In [ ]:
canonical_glm = fit_glms(runs, root, blocks, model, n_jobs=workers)
first_design = canonical_glm["designs"][0, 0]
display(first_design.head())
figures["Design"] = design_figure(runs[0].frame_times, first_design, REGRESSORS)
plt.show()


## 5. Select HRFs across runs

The default library contains canonical SPM plus **512 continuous Sobol samples** (seed 0).
The six parameter ranges are response delay 3–6 s, undershoot delay 10–16 s, response
dispersion 0.5–1.5, undershoot dispersion 0.5–2.5, response/undershoot ratio 2–8, and onset
delay 0–2 s. Custom HRFs have duration 36 s. Sobol balances coverage of this parameter box;
the nonlinear mapping to curves can still produce similar waveforms.

Set `hrf_n_samples` to a power of two and `hrf_seed` to a nonnegative integer. Use
`hrf_library="expanded"` to reproduce the old 649-candidate grid, or supply `hrf_parameters`
to use explicit custom rows. The next cell only generates and plots the library; fitting
starts in the following cell. The parameter table includes **peak_time**, computed from the maximum of the full HRF on a
0.1-second grid. This differs from the response-delay parameter of the positive gamma component.

Selection predicts a shared mean stimulus response in each held-out run from the other runs,
after conditional nuisance projection. It requires no repeated images; it assumes that a mean
response transfers across runs. Summed prediction errors determine **one winning HRF per
grayordinate**, not an average of parameters and not a separate final HRF for each run.

We select using all runs for the final models, and independently within odd and even runs for
reliability. The half-session evaluations freeze both HRF and mean amplitude from the training
half when predicting the other half. The maximized selection-CV score itself is not an independent
performance estimate. All-run selection uses information from every run.

In [ ]:
if settings["hrf_parameters"] is not None:
    library = HrfLibrary.from_parameters(settings["hrf_parameters"])
elif settings["hrf_library"] == "sobol":
    library = sobol_hrf_library(settings["hrf_n_samples"], seed=settings["hrf_seed"])
elif settings["hrf_library"] == "expanded":
    library = expanded_hrf_library()
else:
    raise ValueError("hrf_library must be 'sobol' or 'expanded'")
print(f"HRF candidates: {len(library.candidates)}; fingerprint: {library.fingerprint}")
display(library.parameter_table.head())

figures["Library"] = library_figure(library)
plt.show()


In [ ]:
selections = select_hrfs(runs, root, blocks, library, n_jobs=workers)
hrf_maps = selection_maps(selections, len(brain))


## 6. Fit the same GLM with per-grayordinate HRFs

The helper uses the same events, confounds, scans, and contrasts as the canonical model:

```python
optimized = fit(glm_data, model, hrf_selection=selection,
                feature_signature=spatial_signature)
delta = task_delta_r2(glm_data, model, optimized)
```

The supplied selection replaces `model.hrf_model`. Optimized results have one design for each
run/HRF group instead of one design per run. Both GLMs export task, RT, and trial-type effects,
variances, t statistics, and z scores. Inference is conditional on the selected HRFs and does not
account for selection uncertainty.

In [ ]:
optimized_glm = fit_glms(runs, root, blocks, model, selections=selections, n_jobs=workers)
glms = {"CanonicalGLM": canonical_glm, "OptimizedGLM": optimized_glm}
summary, figures["GLMComparison"] = glm_comparison(glms)
display(summary)
plt.show()


The comparison map is **optimized full R² minus canonical full R²** at each grayordinate.
This is a descriptive in-sample comparison; the models share data with HRF selection and
the difference can be negative. Within each model, **task ΔR² = full R² − confounds-only R²**
measures the variance added jointly by all three task predictors. It is not an RT-only score.

### 6b. Cortical full-model R²

These lateral and medial views show **pooled full-model BOLD R²**, including the
confounds, for the canonical and optimized GLMs. They use the fitted results above;
no refitting or spatial resampling is performed. This is an in-sample summary,
separate from the held-out trial-encoding scores below.

The notebook finds this subject's **fsLR 32k midthickness** surfaces in fMRIPrep.
Set `surface_meshes={"left": path, "right": path}` to use matching fsLR inflated
surfaces instead; native FreeSurfer or fsaverage meshes do not have the required
vertex ordering. The CIFTI vertex IDs place the values, including medial-wall gaps.
Gray indicates missing/unprocessed vertices; subcortical grayordinates are omitted.
Each figure uses a shared color scale across its models, without thresholding.

Set `surface_maps=False` to skip these figures. If no matching surfaces are found,
only the surface figures are skipped. They are registered for PNG export in section 10.


In [ ]:
from examples.NSD.workflow_surfaces import find_surface_meshes, surface_figure

surface_meshes = None
if settings.get("surface_maps", True):
    surface_meshes = find_surface_meshes(
        prep, settings["subject"], paths=settings.get("surface_meshes"))
    if surface_meshes is None:
        print("Surface plots skipped: no bilateral fsLR 32k meshes found. Set surface_meshes to left/right paths.")
    else:
        settings["surface_meshes"] = {hemi: str(path) for hemi, path in surface_meshes.items()}
        print("Surface meshes:", settings["surface_meshes"])
if surface_meshes is not None:
    figures["GLMR2Surface"] = surface_figure(
        {name: result["r2"][0] for name, result in glms.items()},
        brain, surface_meshes, statistic="r2", title="Conventional GLMs: pooled full-model BOLD R²")
    display(figures["GLMR2Surface"])


## 7. HRF parameters and odd/even reliability

Compare the independently selected odd- and even-run parameters across grayordinates. These
plots show agreement and heterogeneity; spatial correlations alone are not a complete reliability
analysis. Fixed library parameters have no correlation coefficient. NaN means no eligible HRF
or an unprocessed grayordinate, never the canonical HRF. The exports include all-run, odd-run,
and even-run parameter maps, including response delay and full-HRF time to peak.

In [ ]:
agreement, figures["HRFReliability"] = parameter_agreement(
    library, hrf_maps["odd"][0], hrf_maps["even"][0])
display(agreement)
plt.show()

display(pd.DataFrame([
    dict(direction=direction, statistic=name,
         median=np.median(values[np.isfinite(values)]) if np.isfinite(values).any() else np.nan)
    for direction in ("odd_to_even", "even_to_odd")
    for name, values in zip(("selected test R²", "canonical test R²", "test ΔR²"), hrf_maps[direction])
]))

### 7b. Full-HRF agreement and the canonical baseline

Parameter correlations can miss agreement between HRFs with different combinations of
parameters. Here we compare the **complete HRF curves at each grayordinate**, correlating
their time samples on the library's common 0.1-second grid, including the undershoot and
zero-padded tails. We do not shift curves to align their peaks.

We compute three Pearson correlations: **odd versus even**, **odd versus canonical SPM**,
and **even versus canonical SPM**. The canonical comparisons provide a baseline for the
agreement expected from the shared broad HRF shape. Points above the diagonal in the
scatterplots mean the other half's HRF matches the reference half better than the canonical
HRF does. Pearson correlation measures shape and ignores overall amplitude scale and offset;
it is not a measure of BOLD prediction accuracy.

The summary and plots use the same grayordinates, requiring both half-session HRFs to be
defined. Saved maps preserve each pair's own valid locations and use NaN elsewhere. The final
export includes `desc-notebookHRFReliability_stat-curvecorrelation.dscalar.nii`, containing
`odd_even_r`, `odd_canonical_r`, and `even_canonical_r` in that order.

This cell uses the existing `library` and `hrf_maps`; it requires no HRF selection or GLM refit.

In [ ]:
from examples.NSD.hrf_reliability import hrf_curve_correlations

# Compare complete curves without aligning their peaks.
curve_r = hrf_curve_correlations(library, hrf_maps["odd"][0], hrf_maps["even"][0])
curve_summary, figures["HRFCurveReliability"] = curve_agreement(curve_r)
display(curve_summary)
plt.show()


## 8. Single-trial beta series

Every stimulus receives its own coefficient, using the original event table. RT and trial type
remain metadata during beta estimation. In CV mode they then predict those betas in a separate
trial-level OLS model: separate training-run intercepts and shared trial-type and RT slopes.
The `task` coefficient is the pooled training beta mean, used as a prediction reference.
Set `encoding_mode="absolute"` to reproduce the historical shared-intercept objective.

Within each inner fold, select HRFs on training runs, estimate training and validation betas
using candidate fractions for training and fixed OLS targets for validation.
Choose the fraction maximizing pooled inner-CV R² separately at each grayordinate. Ties favor
the larger fraction (less shrinkage). Spatial percentiles summarize the scores only. The score
centers each validation run/feature residual before pooling SSE and within-run SST; negative scores are
retained. Fractional CV measures prediction against fixed, noisy OLS estimates,
which remain distinct from latent ground-truth amplitudes.

Odd/even outer evaluations keep test runs out of HRF and penalty selection. A separate all-run
tune produces final beta images. Fractional ridge penalizes raw trial coefficients;
nuisance coefficients stay unpenalized. The fraction is the coefficient-norm ratio in this
raw coefficient basis after nuisance projection: 1 gives OLS; smaller values shrink more.
Each run/grayordinate gets the alpha needed to achieve its chosen fraction. OLS defines the
fraction and supplies the fixed validation targets. Betas retain native signal units. Missing predictors
exclude trials only from the encoding score, never from beta estimation. The conventional GLM
stage above still requires complete covariates. Rerunning this cell uses the current settings.

Held-out outcomes supply only the scoring offset, never training slopes or returned
predictions. Run intercepts and scoring offsets are exported in `_loss.npz`, with run
labels. Predictors must vary within runs; purely between-run effects are unidentifiable.
Saved outputs were cleared when the encoding default changed; rerun to obtain current results.


In [ ]:
beta_models = {}
ridge_cv = {}
mode = settings.get("ridge_mode", "off" if settings.get("ridge_alpha") is None else "fixed")
if mode not in ("fractional_cv", "cv", "fixed", "off"):
    raise ValueError("ridge_mode must be fractional_cv, cv, fixed, or off")
penalties = {"OLS": 0.0}
if mode == "fixed":
    if settings["ridge_alpha"] is None or settings["ridge_alpha"] <= 0:
        raise ValueError("ridge_alpha must be positive in fixed mode")
    penalties["Ridge"] = settings["ridge_alpha"]
for hrf_name, selected in (("Canonical", None), ("Optimized", selections)):
    for estimator, alpha in penalties.items():
        label = hrf_name + "Trial" + estimator
        beta_models[label] = fit_beta_series(runs, root, blocks, selections=selected,
                                            ridge_alpha=alpha, n_jobs=workers)
if mode in ("fractional_cv", "cv"):
    grid = (dict(fractions=settings["ridge_fractions"]) if mode == "fractional_cv"
            else dict(alphas=settings["ridge_alphas"]))
    from examples.NSD.ridge_workflow import fit_cv_beta_series
    for hrf_name, candidate_library in (("Canonical", None), ("Optimized", library)):
        result = fit_cv_beta_series(
            runs, root, library=candidate_library, **grid,
            encoding_mode=settings["encoding_mode"],
            percentile=settings["ridge_percentile"], block_size=settings["block_size"],
            max_grayordinates=settings["max_grayordinates"], n_jobs=workers,
        )
        ridge_cv[hrf_name] = result
        estimator = "TrialFractionalCV" if mode == "fractional_cv" else "TrialRidgeCV"
        beta_models[hrf_name + estimator] = result["final"]
display(beta_models["OptimizedTrialOLS"]["trial_table"].head())

### 8b. Fraction selection and held-out encoding scores

Each grayordinate selects its own fraction for odd-run training, even-run training, and final
all-run fitting. The tables report how many grayordinates selected each fraction. The curves
summarize candidate R² across grayordinates; no single fraction is selected from these curves.
The outer fractional scores evaluate predictions against fixed OLS test targets. They describe
trial-level beta prediction, separately from the BOLD R² maps above. Fraction 1 is OLS.

In legacy `ridge_mode="cv"`, the tables and curves instead show the global-alpha selection.


In [ ]:
if ridge_cv:
    from examples.NSD.ridge_outputs import tuning_table, tuning_figure
    table = tuning_table(ridge_cv)
    fractional = "fraction" in table
    display(table if fractional else table.loc[table.selected])
    display(tuning_figure(ridge_cv))
    if fractional:
        fig, axes = plt.subplots(1, len(ridge_cv), figsize=(6 * len(ridge_cv), 4), squeeze=False)
        for ax, name in zip(axes[0], ridge_cv):
            counts = table.loc[table["mode"] == name].pivot(
                index="fraction", columns="scope", values="selected_grayordinates")
            counts.plot.bar(ax=ax)
            ax.set(title=name, xlabel="Selected fraction", ylabel="Grayordinates")
        fig.tight_layout()
        figures["FractionSelection"] = fig
        plt.show()
    outer_rows = []
    for name, result in ridge_cv.items():
        for split, evaluated in result["evaluation"].items():
            finite = evaluated["encoding_r2"][np.isfinite(evaluated["encoding_r2"])]
            row = dict(model=name, split=split, grayordinates=len(finite),
                       median_encoding_r2=float(np.median(finite)) if len(finite) else np.nan,
                       p90_encoding_r2=float(np.percentile(finite, 90)) if len(finite) else np.nan)
            if not fractional:
                row["alpha"] = evaluated["ridge_alpha"]
            outer_rows.append(row)
    display(pd.DataFrame(outer_rows))
else:
    print("Ridge CV is disabled; no inner selection or outer encoding scores.")


### 8c. Cortical beta-series model R²

These maps show the **BOLD time-series fit** of each final beta-series model,
including its confounds. This is distinct from predicting trial betas from trial
variables. The fitted models include OLS and the ridge method enabled above.
Their R² values share a 0–1 color scale, extended below zero if needed. These
in-sample scores should not be used to choose the regularization strength.


In [ ]:
if surface_meshes is not None:
    figures["BetaR2Surface"] = surface_figure(
        {name: result["r2"][0] for name, result in beta_models.items()},
        brain, surface_meshes, statistic="r2", title="Beta-series models: pooled full-model BOLD R²")
    display(figures["BetaR2Surface"])


## 9. Reaction-time associations

Correlations use trial betas and RT after centering both within each run, excluding nonpositive
or missing RTs. For CV-tuned betas, RT participated in penalty selection: all-run correlations
are descriptive. The outer encoding scores above use test runs excluded from both HRF and
penalty selection.

The plots below retain the OLS comparison: a vertex is chosen by the absolute **odd-run
canonical OLS** correlation and its **even-run** trials are displayed. The canonical plot keeps
those sets separate. The optimized OLS plot uses an all-run HRF and is not an independent
validation. Conventional GLMs remain OLS regardless of the beta-series ridge mode.

In [ ]:
rt_summary = []
for name, result in beta_models.items():
    for scope in ("all", "odd", "even"):
        values = result["rt"][scope]
        finite = values[np.isfinite(values)]
        rt_summary.append(dict(model=name, runs=scope, n=len(finite),
                               median_abs_r=np.median(np.abs(finite)) if len(finite) else np.nan))
display(pd.DataFrame(rt_summary))

odd_r = beta_models["CanonicalTrialOLS"]["rt"]["odd"]
eligible = np.flatnonzero(np.isfinite(odd_r) & (brain.vertex >= 0))
fig, axes = plt.subplots(1, 2, figsize=(10, 3), layout="constrained")
if len(eligible):
    vertex = int(eligible[np.argmax(np.abs(odd_r[eligible]))])
    for ax, name in zip(axes, ("CanonicalTrialOLS", "OptimizedTrialOLS")):
        x, y = even_run_points(beta_models[name]["betas"],
                              [r.events.response_time.to_numpy() for r in runs],
                              [r.number for r in runs], vertex)
        ax.scatter(x, y, s=10, alpha=0.4)
        ax.set(title=f"{name}: grayordinate {vertex}", xlabel="Within-run RT deviation (s)",
               ylabel="Within-run beta deviation")
else:
    for ax in axes:
        ax.text(0.5, 0.5, "No eligible cortical vertices", ha="center")
fig.suptitle("Even-run trials at an odd-run-selected vertex; optimized HRF uses all runs")
figures["RTCheck"] = fig
plt.show()

### 9b. Cortical response-time correlations

The maps show **signed all-run Pearson correlations** between single-trial betas
and RT, with both centered within run. They use the same correlations saved in
the beta-series RT CIFTIs. Red is positive and blue is negative; all models in the
figure share a symmetric color scale set by their largest absolute cortical
correlation. Values are unthresholded and are not significance maps.

These all-run associations are descriptive. RT contributed to CV ridge selection,
and optimized HRFs use all runs. The independent outer encoding scores are reported
separately above. Missing values are gray; volumetric structures are omitted.


In [ ]:
if surface_meshes is not None:
    figures["RTSurface"] = surface_figure(
        {name: result["rt"]["all"] for name, result in beta_models.items()},
        brain, surface_meshes, statistic="rt", title="Beta-series response-time correlations: all runs")
    display(figures["RTSurface"])


## 10. Save maps, designs, and provenance

All files use `desc-notebook...`, leaving earlier script results intact. The output includes:

- Conventional contrast maps (`effects`, `variances`, `t`, `z`), each containing task, RT, and trial type.
- R² maps containing full, confounds-only, and task-added R², plus the signed GLM comparison map.
- All/odd/even HRF indices, parameters, and selection scores; independent half-session prediction scores.
- The complete HRF parameter table and sampled curves, including full-HRF time to peak.
- Per-run beta-series CIFTIs, trial tables, R² and RT maps for each single-trial model.
- In CV mode: candidate score maps/tables, common masks, fold HRF assignments and losses,
  outer predictions/scoring targets, training coefficients, predictor tables, and chosen penalties.
- Fractional CV: selected-fraction CIFTIs for each scope and per-run alpha CIFTIs for outer/final fits.
- Cortical surface PNGs for conventional and beta-series full R² and beta–RT correlations, when surfaces are available.
- Designs grouped by run/HRF in NPZ files, original scan indices/times, transformed events,
  model settings, source identities, selection provenance, and the figures above.

The saved library table maps each HRF ID to its parameters; the curve NPZ stores `times` and
`curves[hrf_id]`. Keep the library fingerprint, spatial ordering, and index map together for reuse.
Design NPZ files contain `run-XX_hrf-YYYY` matrices with `_columns` and `_frame_times` companions;
they load with `np.load(path, allow_pickle=False)`. The metadata JSON records scan trimming,
regressor coding, score definitions, and library settings (method, Sobol count, and seed). HRF selection provenance records training/test folds.

In [ ]:
paths = save_workflow(runs, root, output, library, selections, glms, beta_models,
                      settings=settings, figures=figures, ridge_cv=ridge_cv,
                      subject=settings["subject"], session=settings["session"])
print(f"Saved {len(paths)} files under {output}")
display(pd.DataFrame({"GLM maps": [str(p) for p in paths
                                    if "GLM" in p.name and p.name.endswith(".dscalar.nii")]}))